# Étape 5 — Le modèle a-t-il appris, ou recopié ?

Une LoRA entraînée sur 29 images peut très bien ne rien avoir appris et se
contenter de restituer ce qu'elle a vu. Le score de style monterait quand même,
puisqu'une copie ressemble forcément au jeu d'entraînement. Ce notebook répond
à trois questions, dans l'ordre où elles se posent.

**Q1 — À partir de quelle ressemblance peut-on parler de recopie ?**
On ne fixe pas le seuil à la main. Les 8 images du holdout sont de vraies photos
de zellige que le modèle n'a jamais vues. On mesure à quel point chacune
ressemble à l'image d'entraînement la plus proche : c'est la ressemblance
normale entre deux photos indépendantes du même artisanat. Une image générée qui
reste dans cette fourchette ne copie pas plus que deux carreaux différents ne se
ressemblent entre eux.

**Q2 — Quelles images générées s'approchent le plus de l'entraînement ?**
On les classe, on affiche les pires paires côte à côte, et on juge à l'œil. Un
chiffre seul ne distingue pas une copie d'un motif géométrique commun.

**Q3 — Le modèle a-t-il oublié le reste du monde ?**
On compare la baseline et la LoRA sur les prompts hors domaine. Si la LoRA les a
abîmés, elle a sur-appris : elle ne sait plus faire que du zellige.

**Durée : environ 20 minutes.** Il faut un GPU T4 et le fichier `runs_r8.zip`.

> **Exécution → Modifier le type d'exécution → T4 GPU** avant de commencer.

## 1. Environnement

In [ ]:
# torchao 0.10 est preinstalle sur Colab et peft exige >= 0.16 :
# sans cette desinstallation, le branchement de la LoRA echoue.
!pip -q uninstall -y torchao 2>&1 | tail -1

!pip -q install 'diffusers>=0.27' 'peft>=0.10' 'transformers>=4.38' \
    accelerate safetensors bitsandbytes lpips pyyaml 2>&1 | tail -2

import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())

## 2. Dépôt et jeu de données

On reconstruit le jeu de données à partir de `licenses.csv` : les images ne sont
pas dans Git, seules les informations qui permettent de les retélécharger à
l'identique le sont. Le holdout est indispensable ici — c'est lui qui donne le
seuil de la question Q1.

In [ ]:
!git clone -q https://github.com/HatimTj/lora-zellige-patrimoine.git /content/projet 2>/dev/null || (cd /content/projet && git fetch -q && git reset -q --hard origin/main)
%cd /content/projet
!python src/collect_wikimedia.py --reconstruire \
  && python src/prepare_data.py \
  && python src/legender.py

In [ ]:
from pathlib import Path

n_train = len(list(Path('data/processed/train').glob('*.jpg')))
n_holdout = len(list(Path('data/processed/holdout').glob('*.jpg')))
print(f'{n_train} images d entrainement, {n_holdout} en holdout')
assert n_train > 20, 'jeu d entrainement incomplet'
assert n_holdout >= 5, ('holdout trop petit : sans lui on ne peut pas etablir '
                        'le seuil de recopie, qui est le coeur de cette etape')

In [ ]:
import subprocess, time

DEBUT = time.time()

def lancer(commande):
    """Execute une commande, affiche le temps ecoule, s'arrete si elle echoue."""
    t = time.time()
    print('>>', commande, flush=True)
    code = subprocess.run(commande, shell=True).returncode
    assert code == 0, f'echec (code {code}) : {commande}'
    print(f'   [{(time.time()-t)/60:.1f} min pour cette etape, {(time.time()-DEBUT)/60:.0f} min au total]', flush=True)

## 3. Récupérer la LoRA de l'Étape 3

Glisse **`runs_r8.zip`** dans le panneau Fichiers (icône dossier à gauche), puis
exécute la cellule. Elle cherche l'archive toute seule.

In [ ]:
from pathlib import Path
import shutil, zipfile

FINAL = Path('runs/r8/pytorch_lora_weights.safetensors')
Path('runs/r8').mkdir(parents=True, exist_ok=True)

def installer(archive):
    """Decompresse une archive de poids a la racine du projet."""
    zipfile.ZipFile(archive).extractall('.')
    print('decompresse :', archive)

if FINAL.exists():
    print('poids deja presents')
else:
    # On cherche l'archive la ou Colab depose les fichiers glisses
    trouvees = sorted(Path('/content').glob('*.zip')) + \
               sorted(Path('.').glob('*.zip'))
    if trouvees:
        installer(trouvees[0])
    else:
        # Repli interactif : interrompt « Tout executer »
        print('aucune archive trouvee, ouverture du televersement...')
        from google.colab import files
        for nom in files.upload():
            if nom.endswith('.zip'):
                installer(nom)
            elif nom.endswith('.safetensors'):
                shutil.move(nom, FINAL)

assert FINAL.exists(), 'poids de la LoRA introuvables'
print(f'{FINAL}  ({FINAL.stat().st_size/1e6:.1f} Mo)')

## 4. Régénérer les images à examiner

Les images ne sont pas versionnées dans Git, seuls les `manifeste.json` et
`evaluation.json` le sont. Les graines étant figées, cette regénération redonne
exactement les mêmes images que celles qui ont produit les chiffres de
l'Étape 3 : les mesures déjà enregistrées restent valables.

Environ 9 minutes.

In [ ]:
besoin_base = not Path('results/baseline/D1_s1234.png').exists()
besoin_lora = not Path('results/r8_1000pas/D1_s1234.png').exists()

if besoin_base:
    lancer('python src/generate.py --sortie results/baseline')
else:
    print('images de la baseline deja presentes')

if besoin_lora:
    lancer('python src/generate.py --lora runs/r8 --intensite 1.0 '
           '--sortie results/r8_1000pas')
else:
    print('images de la LoRA deja presentes')

## 5. L'analyse

`overfit_check.py` répond aux trois questions et écrit
`results/surapprentissage.json` plus trois figures.

In [ ]:
lancer('python src/overfit_check.py --lora results/r8_1000pas '
       '--base results/baseline --n-paires 8')

## 6. Q1 — Le seuil de recopie

Deux nuages de points sur le même axe. En haut, les 8 photos du holdout : de
vraies images de zellige que le modèle n'a jamais vues, placées selon leur
ressemblance à l'entraînement. En bas, les images générées.

**Comment lire.** Si les deux nuages se superposent, les images générées ne
ressemblent pas plus à l'entraînement que deux photos distinctes de zellige ne
se ressemblent : c'est du style, pas de la copie. Les points générés qui
dépassent la barre sont les seuls à examiner de près.

In [ ]:
from IPython.display import Image as Afficher
Afficher('results/figures/memorisation_seuil.png')

## 7. Q2 — Les paires les plus ressemblantes

En haut l'image générée, en dessous l'image d'entraînement dont elle est la plus
proche, classées de la plus ressemblante à la moins. Le chiffre est la
similarité DINOv2 ; il est rouge au-dessus du seuil.

**Comment lire.** Une vraie recopie se reconnaît à la structure : même nombre de
branches à l'étoile, même découpe, mêmes défauts du carreau. Deux zelliges
différents qui partagent une palette et une symétrie à 8 branches donnent un
score élevé sans qu'il y ait copie — c'est précisément ce que la géométrie
islamique produit. C'est pour ça qu'on regarde, au lieu de se fier au seul
chiffre.

In [ ]:
Afficher('results/figures/plus_proches_voisins.png')

## 8. Q3 — Le modèle sait-il encore faire autre chose ?

Même prompt, même graine, modèle de base en haut et LoRA en dessous, sur les
trois prompts hors domaine. On montre pour chaque prompt le cas le **plus
défavorable** à la LoRA, pas le plus flatteur.

**Comment lire.** Un CLIPScore qui s'effondre ou un chien qui se couvre de
carreaux signalent un surapprentissage : la LoRA s'est imposée partout au lieu
de rester attachée au token `zlj_style`.

In [ ]:
Afficher('results/figures/hors_domaine.png')

## 9. Le verdict chiffré

In [ ]:
import json

r = json.load(open('results/surapprentissage.json'))
s, v, d = r['seuil_de_recopie'], r['verdict'], r['degradation_hors_domaine']

print('Q1  seuil de recopie, mesure sur le holdout')
print(f'    {s["n_holdout"]} vraies photos jamais vues contre {s["n_train"]} d entrainement')
print(f'    ressemblance maximale moyenne : {s["similarite_max_moyenne"]}')
print(f'    seuil (le plus eleve)         : {s["seuil"]}')

print('\nQ2  images generees au-dessus du seuil')
print(f'    {v["images_au_dessus_du_seuil"]} / {v["images_examinees"]}  ({v["part_au_dessus_pct"]} %)')
print(f'    similarite maximale observee  : {v["similarite_max_observee"]}')

print('\nQ3  prompts hors domaine')
print(f'    CLIPScore {d["clipscore_base"]} -> {d["clipscore_lora"]}  ({d["ecart_moyen"]:+})')
print(f'    pire ecart sur une image      : {d["pire_ecart"]:+}')

## 10. Sauvegarder

On pousse les figures, le JSON d'analyse et le nouvel outil. Les images restent
hors de Git, comme toujours.

In [ ]:
from google.colab import userdata
jeton = userdata.get('GITHUB_TOKEN')
url = f'https://{jeton}@github.com/HatimTj/lora-zellige-patrimoine.git'

!git config user.name 'HatimTj'
!git config user.email 'hatimtajimi1@gmail.com'
!git add results/surapprentissage.json results/figures/*.png
!git add results/*/manifeste.json
!git commit -m 'etape5: analyse du surapprentissage'
!git pull --rebase -q {url} main
!git push {url} main